# Package Installation

In [ ]:
!pip install -U peft datasets sacrebleu sentencepiece accelerate evaluate bitsandbytes transformers==4.44.0

In [ ]:
import pandas as pd
import sacrebleu
import numpy as np
import torch
import transformers
import peft
import datasets
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


print("torch", torch.__version__)
print("transformers", transformers.__version__, "| peft", peft.__version__)
print("datasets", datasets.__version__)
print("GPU count:", torch.cuda.device_count())

In [ ]:
from huggingface_hub import login
hf_token = "your_token"
login(token=hf_token)

# Load data

In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd

base_path = "/kaggle/input/vlsp-dataset/data"

def read_parallel(src_file, tgt_file):
    with open(base_path + src_file, encoding="utf-8") as f_src, open(base_path + tgt_file, encoding="utf-8") as f_tgt:
        src = f_src.read().strip().splitlines()
        tgt = f_tgt.read().strip().splitlines()
    n = min(len(src), len(tgt))
    return pd.DataFrame({'en': src[:n], 'vi': tgt[:n]})

train_df = read_parallel("/train.en.txt", "/train.vi.txt")
test_df  = read_parallel("/public_test.en.txt", "/public_test.vi.txt")
train_df = train_df.drop_duplicates().reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

raw = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})
print(raw)

In [ ]:
example_en = train_df.iloc[0]['en']
example_vi = train_df.iloc[0]['vi']
print(example_en, example_vi, sep='\n\n')

# Prepare model

In [ ]:
MODEL_NAME = "VietAI/envit5-translation"

In [ ]:
from transformers import AutoModelForSeq2SeqLM

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
model.config.use_cache = False

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# LoRA config

In [ ]:
import re
def get_num_layers(model):
    numbers = set()
    for name, _ in model.named_parameters():
        for number in re.findall(r'\d+', name):
            numbers.add(int(number))
    return max(numbers)

def get_last_layer_linears(model):
    names = []
    
    num_layers = get_num_layers(model)
    for name, module in model.named_modules():
        if str(num_layers) in name and not "encoder" in name:
            if isinstance(module, torch.nn.Linear):
                names.append(name)
    return names

In [ ]:
get_num_layers(model)

In [ ]:
get_last_layer_linears(model)

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["k","q","v","o"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

In [ ]:
model

# Generate example

In [ ]:
example_en = train_df.iloc[0]['en']
example_vi = train_df.iloc[0]['vi']
print(example_en, example_vi, sep='\n\n')

In [ ]:
def translate(text, model, tokenizer):
    prefix = "en: "
    input_text = prefix + text
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs, 
            max_length=512, 
            num_beams=5,
            early_stopping=True
        )
    
    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(f"en: {example_en}")
model.eval()
print(f"{translate(example_en, model, tokenizer)}")

# Format finetune data

In [ ]:
def preprocess_function(examples):
    inputs = ["en: " + doc for doc in examples["en"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    targets = ["vi: " + doc for doc in examples["vi"]]
    labels = tokenizer(text_target=targets, max_length=512, truncation=True)

    labels_input_ids = labels["input_ids"]
    labels_input_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels_input_ids
    ]
    
    model_inputs["labels"] = labels_input_ids
    return model_inputs

tokenized_raw = raw.map(
    preprocess_function,
    batched=True,
    remove_columns=raw["train"].column_names
)

In [ ]:
print(tokenized_raw["train"][0].keys())

In [ ]:
print(tokenized_raw["train"][0])

# Train

In [ ]:
import evaluate
import numpy as np

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8
)
metric = evaluate.load("sacrebleu")

def postprocess(preds, labels):
    preds = [p.strip() for p in preds]
    preds = [p.replace("vi:", "").replace("vi :", "").strip() for p in preds]
    
    labels = [l.strip() for l in labels]
    labels = [[l.replace("vi:", "").replace("vi :", "").strip()] for l in labels]
    
    preds = [' '.join(p.split()) for p in preds]
    labels = [[' '.join(l[0].split())] for l in labels]
    
    return preds, labels

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    preds = np.clip(preds, 0, len(tokenizer) - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds, decoded_labels = postprocess(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
targs = Seq2SeqTrainingArguments(
    output_dir="./finetuned_model",
    fp16=True,       
    bf16=False,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, 
    learning_rate=2e-4,
    num_train_epochs=1,    
    logging_steps=250,
    save_strategy="steps",
    eval_strategy="epoch",
    save_steps=250,        
    save_total_limit=2,
    report_to="none",
    group_by_length=True, 
    ddp_find_unused_parameters=False,
    dataloader_num_workers=4,
    remove_unused_columns=False,
    predict_with_generate=True,
    generation_max_length=512
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=targs,
    train_dataset=tokenized_raw["train"],
    eval_dataset=tokenized_raw["test"], 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
print("Start training...")
trainer.train()

In [ ]:
for name, param in model.named_parameters():
    if "lora_A" in name:
        print(name, param.data.abs().mean().item())
        break

In [ ]:
def translate(text, model, tokenizer):
    prefix = "en: "
    input_text = prefix + text
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs, 
            max_length=512, 
            num_beams=5,
            early_stopping=True
        )
    
    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(f"en: {example_en}")
model.eval()
print(f"{translate(example_en, model, tokenizer)}")

# Save model

In [ ]:
trainer.save_model("/kaggle/working/final_translation_model")
tokenizer.save_pretrained("/kaggle/working/final_translation_model")